In [ ]:
"""
Modify the baseline fuelscape (Landscape-scale)
Landscape-scale means we are "treating" every pixel in the fuelscape
Maxwell.Cook@colostate.edu
"""

# standard imports
import os, sys

# custom functions
sys.path.append(os.getcwd()) # add code folder to system path
from code.Python.archive.__functions import *  # imports all custom functions

# directories
# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

# environ vars
proj_crs = 'EPSG:26913' # NAD83 UTM Zone 13N

print("Ready to go !")

In [ ]:
print("exe:", sys.executable)
print("prefix:", sys.prefix)
print("PATH head:", os.environ.get("PATH","")[:250])

In [ ]:
import rasterio
import rioxarray
print("rasterio", rasterio.__version__)
print("rioxarray", rioxarray.__version__)
rasterio.show_versions()

In [ ]:
# --- Load the fuel adjustment tables
canopy = pd.read_csv(os.path.join(projdir, 'data/tabular/fuel_mods/canopy_effects.csv'))
surface = pd.read_csv(os.path.join(projdir, 'data/tabular/fuel_mods/surface_effects.csv'))
print(f"Canopy effects:\n{canopy}")
print(f"\nSurface effects:\n{surface}")

In [ ]:
print(f"Surface treatment types:\n\t{surface.columns}")
print(f"\nCanopy treatment types:\n\t{canopy['Treatment'].unique()}")

In [ ]:
# --- Create the scenario lookup table once
scenarios = {
    'Hand Thin': {'canopy': 'Hand Thin', 'surface': 'Manage'}, # Hand thin + removal
    'Mech Thin': {'canopy': 'Mech Thin', 'surface': 'Manage'}, # Mech thin + removal
    'RxFire': {'canopy': 'RxFire', 'surface': 'RxFire'}, # Broadcast Burn
    'Complete Hand': {'canopy': 'Complete Hand', 'surface': 'RxFire'}, # Complete Hand
    'Complete Mech': {'canopy': 'Complete Mech', 'surface': 'RxFire'}, # Complete Mech
    'Masticate': {'canopy': 'Masticate', 'surface': 'Mastication'}, # Mastication
    'Patch': {'canopy': 'Patch', 'surface': 'Slash'}, # Patch-cut with scatter
    'Patch_SM': {'canopy': 'Patch', 'surface': 'Patch_SM'}, # Patch-cut with surface removal
    'LowSevWF': {'canopy': 'LowSevWF', 'surface': 'LowSevWF'}, # Low-severity wildfire
    'ModSevWF': {'canopy': 'ModSevWF', 'surface': 'ModSevWF'}, # Moderate-severity wildfire
    'HighSevWF': {'canopy': 'HighSevWF', 'surface': 'HighSevWF'} # High-severity wildfire
}

print(f"Treatment scenarios:\n\t{scenarios.keys()}")

In [ ]:
def build_fbfm_lut(surface_df, scenario_col, fm='FBFM40'):
    """
    Build a numpy lookup table from the surface fuel adjustment table

    :param surface_df:
    :param scenario_col:
    :param baseline_col:
    :return:
    """
    base = surface_df[fm].to_numpy()
    new  = surface_df[scenario_col].to_numpy()

    vmax = int(np.nanmax(base)) # find the maximum fuel model code
    lut = np.arange(vmax + 1, dtype=np.int16)  # identity by default (e.g., 201 = 201)
    lut[base.astype(int)] = new.astype(np.int16)
    return lut

def apply_fuel_adjustments(
        lcp, # the landscape file (baseline)
        canopy_lut, # the canopy fuel adjustment lookup table
        surface_lut, # the surface fuel adjustment lookup table
        scenario_key, # the treatment type
        scenarios, # mapping dictionary for treatment -> adjustments
        BANDS # band lookup table
):
    # --- Gather in inputs
    trt = scenarios[scenario_key] # the treatment type and associated adjustments
    canopy_nm = trt['canopy'] # the canopy type (e.g., Mech Thin)
    surface_nm = trt['surface'] # the surface type (e.g., Manage)

    out_da = lcp.copy(deep=True) # copies the coords & attributes of the fuelscape

    # --- 1. Surface fuel remapping
    assert surface_nm in surface_lut.columns, \
        f"Surface fuel scenario not found: {surface_nm}"

    rmp = build_fbfm_lut(surface_lut, surface_nm) # build the remap table
    fm = out_da.sel(band=BANDS['FBFM40']).astype(np.int32) # get the fuel model band
    fm2 = rmp[fm].astype(np.int16) # get the remap value
    out_da.loc[dict(band=BANDS['FBFM40'])] = fm2 # replace those values

    # --- 2. Canopy fuel adjustments
    canopy_lut = canopy_lut.set_index('Treatment') # set treatment type as the index
    assert canopy_nm in canopy_lut.index, \
        f"Canopy fuel scenario not found: {canopy_nm}"

    # gather the adjustment factors
    r = canopy_lut.loc[canopy_nm] # get the treatment index
    print(f"\t\tCanopy fuel adjustments:\n{r}")

    # --- Build the band lookup dictionary
    bands = ['CC','CH','CBD','CBH']
    afs = ['cc_AF','ch_AF','cbd_AF','cbh_AF']
    canopy_bands = {b: lcp.sel(band=BANDS[b]) for b in bands}

    # create a post-treatment forest mask
    cc2 = np.floor(canopy_bands['CC'] * float(r['cc_AF']))
    is_forest = cc2 >= 10

    # apply the adjustment factors
    for b, af in zip(bands, afs):
        arr = np.floor(
            canopy_bands[b].astype(np.float32) * float(r[af])
        ).astype(np.int16) # get the correct band and adjustment
        arr = xr.where(is_forest, arr, 0) # handle the forest mask
        out_da.loc[dict(band=BANDS[b])] = arr # assign back to the multiband stack

    return out_da

# --- Load the baseline fuelscape
lcp = list_files(projdir, "fuelscape_baseline.tif", recursive=True)
lcp = rxr.open_rasterio(lcp[0])
bands_lut = (
    dict(zip(lcp.attrs.get("long_name", None),
             lcp["band"].values.tolist()))
)
print(bands_lut)

# --- Create the output directory
out_dir = join(projdir, f'data/spatial/fuelscape/NCFC/treated/landscape/')
os.makedirs(out_dir, exist_ok=True)

# --- Run the treatment scenarios
out_fsc = {}
for scen in scenarios.keys():
    scen_ = scen.replace(" ", "_")
    print(f"Creating fuelscape for: {scen}")

    # run the fuelscape adjustment
    treated = apply_fuel_adjustments(
        lcp = lcp,
        canopy_lut = canopy,
        surface_lut = surface,
        scenario_key = scen,
        scenarios = scenarios,
        BANDS = bands_lut
    )
    # save the treated fuelscape
    treated.rio.to_raster(
        join(out_dir, f'{scen_}.tif'),
        tiled=True,
        compress="DEFLATE",
        BIGTIFF="YES",
    )
    out_fsc[scen] = treated

# Check on one
out_fsc['ModSevWF']

## Load the fuelscape raster (multi-band geotiff)

** Reference 01-Baseline_Fuelscape.ipynb **

The baseline fuelscape is a multi-band geotiff representing the landscape topography and pre-treatment fuel conditions. It includes all bands needed to run FlamMap models for fire behavior outputs. Notably, this baseline fuelscape also includes CFRI-specific adjustments to the lodgepole pine EVT to better reflect expected fire behavior in this system. This adjustment was done in the downloading of the fuelscape from the LFPS REST API.

In [ ]:
# load the baseline fuelscape raster
lcp = list_files(projdir, "fuelscape_baseline.tif", recursive=True)
lcp = rxr.open_rasterio(lcp[0])
print(lcp)

In [ ]:
lcp.rio.crs

In [ ]:
BANDS = {
    "ELEV":   1,
    "SLP":    2,
    "ASP":    3,
    "FBFM40": 4,
    "CC":     5,
    "CH":     6,
    "CBH":    7,
    "CBD":    8,
}

cc = lcp.sel(band=BANDS["CC"])
float(cc.max()), float(cc.min())

In [ ]:
# make the adjustments to fuelscape
# define the bands (CC, CH, CBD, CBH)
band_map = {
    'cc_AF': 5,
    'ch_AF': 6,
    'cbh_AF': 7,
    'cbd_AF': 8,
}
modified_lcp = lcp.copy() # work with a copy

# map the bands, make the adjustments
for key, band_idx in band_map.items():
    print(f"Processing {key}/{band_idx}")
    # isolate the current raster band
    band = modified_lcp.sel(band=band_idx)
    # map the treatment codes
    for code, treatment in code_lookup.items():
        print(f"\tWorking on {code}/{treatment}")
        # make sure it is a valid treatment
        if treatment not in effect_dict:
            print(f"\t[INFO] Skipping treatment '{treatment}' (no effect data)")
            continue
        # create the treatment mask
        mask = (treat_arr == code)
        factor = effect_dict[treatment][key]
        # multiply the band by the correct factor
        band = band.where(~mask, np.floor(band * factor)) # ensure downward rounding with np.floor

    # assign new band to the stack
    modified_lcp.loc[dict(band=band_idx)] = band

    # check the values
    before = lcp.sel(band=band_idx).values
    after = modified_lcp.sel(band=band_idx).values
    print(f"Δ {key}: {(before != after).sum()} pixels changed")

# ensure the output has the correct band types
modified_lcp # check it

### Surface Modifications

Here we adjust surface fuel codes based on treatment types. The "surface_effects.csv" provides a lookup table for the adjustments to the FBFM40 fuel model based on the given treatment type. We need to adjust FBFM40 values for overlapping treatment polygons, keeping all other bands the same.

In [ ]:
# load the surface fuel modification table
surface_effects = pd.read_csv(os.path.join(projdir, 'data/tabular/fuel_mods/surface_effects.csv'))

# Set FBFM40 as index to create the lookup table
surface_lut = surface_effects.set_index('FBFM40')
# Convert each column (treatment) to a dict: {original_fuel_model: new_fuel_model}
surface_dict = {col: surface_lut[col].to_dict() for col in surface_lut.columns}

surface_effects

In [ ]:
# Filter to surface treatments
surface_trts = trt[(trt['SurfEff'] != 'None') & (~trt['SurfEff'].isnull())].copy()
# set 'Mastication' as rearrange
surface_trts.loc[surface_trts['SurfEff'] == 'Masticate', 'SurfEff'] = 'Rearrange'

# Assign a numeric code for each surface treatment type
surface_trts['treat_code'] = surface_trts['SurfEff'].astype('category').cat.codes
surface_code_lookup = dict(enumerate(surface_trts['SurfEff'].astype('category').cat.categories))
print("\n", surface_code_lookup)

# Rasterize treatment polygons to match fuelscape
surface_arr = rasterize_it(surface_trts, to_img=lcp, attr='treat_code', fill_val=-1)

# save this out to check.
out_fp = os.path.join(projdir, f'data/spatial/treatments/{aoi}/surface_treatments.tif')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
surface_arr.rio.to_raster(out_fp)
print(f"Saved to: {out_fp}")

In [ ]:
# Copy fuelscape to modify
modified_lcp_ = modified_lcp.copy()

# Extract original FBFM40 band (Band 4 = FBFM40)
band_idx = 4
fbfm = modified_lcp_.sel(band=band_idx).copy()

# Loop over treatment types and remap surface fuel codes
for code, treatment in surface_code_lookup.items():
    print(f"Processing surface treatment: {treatment}")
    if treatment not in surface_dict:
        print(f"\t[INFO] No surface mapping found for '{treatment}'")
        continue

    # Treatment mask
    mask = (surface_arr == code)

    # Get remap dict for this treatment
    remap = surface_dict[treatment]

    # Apply remapping to pixels where mask is True
    original_values = fbfm.where(mask).values
    new_values = np.vectorize(remap.get)(original_values)

    # Replace values in the band (where mask)
    fbfm = fbfm.where(~mask, new_values)

# Assign updated band
modified_lcp_.loc[dict(band=band_idx)] = fbfm

print("Surface effects stamped in !")

In [ ]:
# convert to integer
modified_lcp_ = modified_lcp_.astype("int16")
# save the treated fuelscape raster
out_fp = os.path.join(projdir, f'data/spatial/fuelscape/{aoi}/treated/fuelscape_treated.tif')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
modified_lcp_.rio.to_raster(out_fp)
print(out_fp)

## Create a distance to treatment surface

To assess the spatial decay of treatment effects, we can create a euclidean distance raster representing the distance (m) from treatments. Eventually, coming up with a density map as well to represent the clustering of treatments and how they collectively effect fire behavior or reduced risk (i.e., does the spatial pattern of treatments on the landscape matter?).

In [ ]:
# create a binary raster for treated / not treated
trt['treated'] = 1  # flag for treatment
# rasterize the treatment polygons
trt_bin = rasterize_it(
    trt,
    to_img=modified_lcp_,
    attr='treated',
    fill_val=0  # untreated = 0
)

# calculate the Euclidean distance raster
# create a binary mask
mask = (trt_bin.values == 0)

# calculate the Euclidean distance
trt_dist = distance_transform_edt(mask)

# convert to a raster with the appropriate attributes
dist_arr = xr.DataArray(
    trt_dist,
    coords=trt_bin.coords,
    dims=trt_bin.dims,
    name="distance_to_treatment"
)
# write the CRS
dist_arr.rio.write_crs(trt_bin.rio.crs, inplace=True)

# make sure the data align
dist_arr = dist_arr.rio.reproject_match(modified_lcp)

# save the raster out.
out_fp = os.path.join(projdir, f"data/spatial/treatments/{aoi}/distance_to_treatment.tif")
dist_arr.rio.to_raster(out_fp)
print(f"Saved distance raster (meters): {out_fp}")